<a href="https://colab.research.google.com/github/Faucherie/AgileCurrencyProject_midterm/blob/main/cbis_ddsm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CBIS DDSM: Patch Classification

## Table of Contents

- [0 Environment Setup](#scrollTo=QU60D2KNpAEW)
- [1 Data Acuisition](#scrollTo=V8eFyqBznoF7)

## 0. Environment Setup

**Sections**
- [0.1 Install and Import](#scrollTo=FGvdtBenpW6k)
- [0.2 Mount Google Drive](#scrollTo=lz0uS2zPpb_R)

### 0.1 Install and Import

In [ ]:
# The public DICOM annotations/results Access Routing for TCIA
%pip install idc_index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 132.6 MB/s eta 0:00:00


In [ ]:
import os
import sys
import pandas as pd
from idc_index import IDCClient

### 0.2 Mount Google Drive

In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Create a folder in Drive for as the dir for raw CBIS-DDSM files. This assumes the files `CM3070/CBIS_DDSM_Raw/` exsists in the `MyDrive` home folder of my Google Drive account.

In [ ]:
download_path = '/content/drive/MyDrive/CM3070/CBIS_DDSM_Raw'
os.makedirs(download_path, exist_ok=True)
print(f"Target directory set to: {download_path}")

Target directory set to: /content/drive/MyDrive/CM3070/CBIS_DDSM_Raw


## 1 Data Acuisition

### Sections
- [1.1 Explore the IDC Index for CBIS-DDSM](#scrollTo=O0OA9ZGsaM0v)
- [1.2 Collection Summary](#scrollTo=Kax8Cp9qZUbj)
- [1.3 Query CBIS-DDSM](#scrollTo=J7jUXFIp1Hlo)
  - [1.3.1 Sex of patients](#scrollTo=TA-eGfcm7kBl)
  - [1.3.2 Age of patients](#scrollTo=7VTMXBXcOoTM)
- [1.4 Download the DICOM](#scrollTo=bvK0sXey-P-0)

### References

- [Lee, R. S., Gimenez, F., Hoogi, A., Miyake, K. K., Gorovoy, M., & Rubin, D. L. (2017). A curated mammography data set for use in computer-aided detection and diagnosis research. *Scientific Data*, 4, 170177](https://doi.org/10.1038/sdata.2017.177)
- [The Cancer Imaging Archive CBIS-DDSM | Curated Breast Imaging Subset of Digital Database for Screening Mammography](https://www.cancerimagingarchive.net/collection/cbis-ddsm/)
- [Github: ImagingDataCommons idc-index](https://github.com/ImagingDataCommons/idc-index)
- [TCIA API guide](https://www.cancerimagingarchive.net/tcia-api-guides/)
- [`tcia-utils` package](https://pypi.org/project/tcia-utils/)
- [Example notebook demonstrating tcia_utils functionality for TCIA Public DICOM data](https://github.com/kirbyju/TCIA_Notebooks/blob/main/Accessing_TCIA_Public_DICOM_data_in_IDC.ipynb)


### 1.1 Explore the IDC Index for CBIS-DDSM

In [32]:
idc_client = IDCClient()

In [34]:
client_columns = idc_client.index.columns.values
print(client_columns)

['collection_id' 'analysis_result_id' 'PatientID' 'SeriesInstanceUID'
 'StudyInstanceUID' 'source_DOI' 'PatientAge' 'PatientSex' 'StudyDate'
 'StudyDescription' 'BodyPartExamined' 'Modality' 'SOPClassUID'
 'sop_class_name' 'TransferSyntaxUID' 'transfer_syntax_name'
 'PhotometricInterpretation' 'PixelRepresentation' 'Manufacturer'
 'ManufacturerModelName' 'SeriesDate' 'SeriesDescription' 'SeriesNumber'
 'instanceCount' 'license_short_name' 'series_init_idc_version'
 'series_revised_idc_version' 'aws_bucket' 'crdc_series_uuid'
 'series_aws_url' 'series_size_MB']


List all collections available through IDC, to confirm `cbis_ddsm` is present.

In [ ]:
collections = idc_client.index['collection_id'].unique()

print("Available Collections:")
display(collections)

Available Collections:


array(['4d_lung', 'acrin_6698', 'acrin_contralateral_breast_mr',
       'acrin_flt_breast', 'acrin_nsclc_fdg_pet', 'adrenal_acc_ki67_seg',
       'advanced_mri_breast_lesions', 'anti_pd_1_lung',
       'b_mode_and_ceus_liver', 'bonemarrowwsi_pediatricleukemia',
       'breast_cancer_screening_dbt', 'breast_diagnosis',
       'breast_mri_nact_pilot', 'c4kc_kits', 'catch', 'cbis_ddsm',
       'cc_radiomics_phantom', 'cc_radiomics_phantom_2',
       'cc_radiomics_phantom_3', 'cc_tumor_heterogeneity', 'ccdi_mci',
       'cddp_eagle_1', 'cgci_blgsp', 'cgci_htmcp_cc', 'cgci_htmcp_dlbcl',
       'cgci_htmcp_lc', 'cmb_aml', 'cmb_brca', 'cmb_crc', 'cmb_gec',
       'cmb_lca', 'cmb_mel', 'cmb_mml', 'cmb_ov', 'cmb_pca', 'cmmd',
       'colorectal_liver_metastases', 'covid_19_ar', 'covid_19_ny_sbu',
       'cptac_aml', 'cptac_brca', 'cptac_ccrcc', 'cptac_cm', 'cptac_coad',
       'cptac_gbm', 'cptac_hnscc', 'cptac_lscc', 'cptac_luad', 'cptac_ov',
       'cptac_pda', 'cptac_sar', 'cptac_stad', 'cpt

Filter the index down to the `cbis_ddsm` collection.

In [ ]:
cbis_df = idc_client.index[idc_client.index['collection_id'] == 'cbis_ddsm']
display(cbis_df.head())

,collection_id,analysis_result_id,PatientID,SeriesInstanceUID,StudyInstanceUID,source_DOI,PatientAge,PatientSex,StudyDate,StudyDescription,...,SeriesDescription,SeriesNumber,instanceCount,license_short_name,series_init_idc_version,series_revised_idc_version,aws_bucket,crdc_series_uuid,series_aws_url,series_size_MB
76379,cbis_ddsm,None,Calc-Test_P_00038_LEFT_CC_1,1.3.6.1.4.1.9590.100.1.2.419081637812053404913...,1.3.6.1.4.1.9590.100.1.2.161465562211359959230...,10.7937/k9/tcia.2016.7o02s9cy,None,None,2017-08-29,None,...,cropped images,1,2,CC BY 3.0,22,22,idc-open-data,77040702-ff2e-49bf-86ec-9331ebc3f8e0,s3://idc-open-data/77040702-ff2e-49bf-86ec-933...,14.055890
76380,cbis_ddsm,None,Calc-Test_P_00038_RIGHT_CC_2,1.3.6.1.4.1.9590.100.1.2.360550081712464813321...,1.3.6.1.4.1.9590.100.1.2.248538452013626298441...,10.7937/k9/tcia.2016.7o02s9cy,None,None,2017-08-29,None,...,cropped images,1,2,CC BY 3.0,22,22,idc-open-data,a133e8cc-c870-471c-94ae-ccb0b4f1952d,s3://idc-open-data/a133e8cc-c870-471c-94ae-ccb...,13.238072
76381,cbis_ddsm,None,Calc-Test_P_00038_RIGHT_MLO_1,1.3.6.1.4.1.9590.100.1.2.126295284812046209819...,1.3.6.1.4.1.9590.100.1.2.348569460311013218440...,10.7937/k9/tcia.2016.7o02s9cy,None,None,2017-08-29,None,...,cropped images,1,2,CC BY 3.0,22,22,idc-open-data,e0c6b7b9-0619-45b0-a111-e94cef838f8b,s3://idc-open-data/e0c6b7b9-0619-45b0-a111-e94...,15.170028
76382,cbis_ddsm,None,Calc-Test_P_00038_RIGHT_MLO_2,1.3.6.1.4.1.9590.100.1.2.326152312412979080929...,1.3.6.1.4.1.9590.100.1.2.573626950127809277210...,10.7937/k9/tcia.2016.7o02s9cy,None,None,2017-08-29,None,...,cropped images,1,2,CC BY 3.0,22,22,idc-open-data,78f84c35-b4eb-4e28-819f-ec2ff8b93dc1,s3://idc-open-data/78f84c35-b4eb-4e28-819f-ec2...,14.618428
76383,cbis_ddsm,None,Calc-Test_P_00041_LEFT_MLO_2,1.3.6.1.4.1.9590.100.1.2.399466258212646932018...,1.3.6.1.4.1.9590.100.1.2.372962290011068589008...,10.7937/k9/tcia.2016.7o02s9cy,None,None,2017-08-29,None,...,cropped images,1,2,CC BY 3.0,22,22,idc-open-data,a6922d61-0a1c-4b07-9abf-f88102d649a5,s3://idc-open-data/a6922d61-0a1c-4b07-9abf-f88...,22.926678


### 1.2 Collection Summary

In [ ]:
from enum import unique
class IDC_CBIS_DDSM_Summary:
    unique_source_dois = cbis_df['source_DOI'].nunique()
    unique_licenses = cbis_df['license_short_name'].nunique()
    unique_body_parts = cbis_df['BodyPartExamined'].nunique()
    unique_modalities = cbis_df['Modality'].nunique()
    num_patients = cbis_df['PatientID'].nunique()
    num_studies = cbis_df['StudyInstanceUID'].nunique()
    num_series = cbis_df['SeriesInstanceUID'].nunique()
    total_series_size_mb = cbis_df['series_size_MB'].sum()

In [47]:
summary_dict = {
    "collection_id": "cbis_ddsm",
    "Unique_Source_DOIs": ", ".join(cbis_df['source_DOI'].dropna().unique().tolist()),
    "Unique_Licenses": ", ".join(cbis_df['license_short_name'].dropna().unique().tolist()),
    "Unique_Body_Parts": ", ".join(cbis_df['BodyPartExamined'].dropna().unique().tolist()),
    "Unique_Modalities": ", ".join(cbis_df['Modality'].dropna().unique().tolist()),
    "Num_Patients": cbis_df['PatientID'].nunique(),
    "Num_Studies": cbis_df['StudyInstanceUID'].nunique(),
    "Num_Series": cbis_df['SeriesInstanceUID'].nunique(),
    "Total_Series_Size_MB": cbis_df['series_size_MB'].sum(),
}
for key, value in summary_dict.items():
    print(f"{key}: {value}")

collection_id: cbis_ddsm
Unique_Source_DOIs: 10.7937/k9/tcia.2016.7o02s9cy
Unique_Licenses: CC BY 3.0
Unique_Body_Parts: BREAST, Left Breast, Right Breast
Unique_Modalities: MG
Num_Patients: 6671
Num_Studies: 6775
Num_Series: 6775
Total_Series_Size_MB: 163512.966728


As expected, CBIS-DDSM examines the breast using the mammography (`MG`) modality. There are 6,671 patients and 6,775 studies; most patients have a single study, but some have multiple (e.g. different views, multiple abnormalities, or both breasts examined).

### 1.3 Query CBIS-DDSM IDC-Index Prior to download

#### 1.3.1 Sex of patients

In [48]:
patient_sex_counts = cbis_df['PatientSex'].value_counts(dropna=False)

print("DICOM series counts of each unique PatientSex:")
display(patient_sex_counts)

DICOM series counts of each unique PatientSex:


,count
PatientSex,
None,6775


The IDC allows for `None`,`F`,`M`,`O` and others for their sereis. With only `None` and equal to `Num_Studies` the DDSM didn't collect sex as it is implied as female.

#### 1.3.2 Age of patients

In [49]:
patient_age_counts = cbis_df['PatientAge'].value_counts(dropna=False)

print("DICOM series counts of each unique PatientAge:")
display(patient_age_counts)

DICOM series counts of each unique PatientAge:


,count
PatientAge,
None,6775


The age of the patients are not recorded per patient for the CBIS-DDSM dataset.

### 1.4 Download the DICOM Images

Download the full `cbis_ddsm` collection (~164 GB) directly into the Drive folder `CM3070/CBIS_DDSM_Raw` created in Section 0.2.



In [55]:
help(idc_client.download_from_selection)

Help on method download_from_selection in module idc_index.index:

download_from_selection(downloadDir, dry_run=False, collection_id=None, patientId=None, studyInstanceUID=None, seriesInstanceUID=None, sopInstanceUID=None, crdc_series_uuid=None, quiet=True, show_progress_bar=True, use_s5cmd_sync=False, dirTemplate='%collection_id/%PatientID/%StudyInstanceUID/%Modality_%SeriesInstanceUID', source_bucket_location='aws') method of idc_index.index.IDCClient instance
    Download the files corresponding to the selection. The filtering will be applied in sequence (but does it matter?) by first selecting the collection(s), followed by
    patient(s), study(studies) and series. If no filtering is applied, all the files will be downloaded.

    Args:
        downloadDir: string containing the path to the directory to download the files to
        dry_run: calculates the size of the cohort but download does not start
        collection_id: string or list of strings containing the values of colle

In [ ]:
# dry run

idc_client.download_from_selection(
    collection_id='cbis_ddsm',
    downloadDir=download_path,
    use_s5cmd_sync=True,
    show_progress_bar=True
)

In [ ]:
# Uncomment to load
"""
idc_client.download_from_selection(
    collection_id='cbis_ddsm',
    downloadDir=download_path,
    use_s5cmd_sync=True,
    show_progress_bar=True
)
"""

**Verify the download**

Confrim the we download all 6775 studies

In [ ]:
!find /content/drive/MyDrive/CM3070/CBIS_DDSM_Raw -type f | wc -l

In [54]:
!du -sh /content/drive/MyDrive/CM3070/CBIS_DDSM_Raw

^C


### 1.5 Load TCIA CSVs

These carry the official, patient_id-based train/test split and per-abnormality annotations (pathology, BI-RADS assessment, mass shape/margin or calcification type/distribution) that are not encoded in the DICOM headers themselves. Download them from the [CBIS-DDSM TCIA page](https://www.cancerimagingarchive.net/collection/cbis-ddsm/) under "Data Access".

Load the ground truth CSVs. These carry the official patient_id-based train/test split.

In [53]:
csv_dir = "/content/drive/MyDrive/CM3070/cbis-csvs"
files = {
    "mass_train": "mass_case_description_train_set.csv",
    "mass_test":  "mass_case_description_test_set.csv",
    "calc_train": "calc_case_description_train_set.csv",
    "calc_test":  "calc_case_description_test_set.csv",
}

dfs = {name: pd.read_csv(os.path.join(csv_dir, fname)) for name, fname in files.items()}
for name, df in dfs.items():
    print(f"{name}: {df.shape[0]} rows")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CM3070/cbis-csvs/mass_case_description_train_set.csv'